In [1]:
import sys
sys.path.insert(0, "..")
import mlflow.tensorflow
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
from recommenders.evaluation.python_evaluation import (
    map_at_k,
    r_precision_at_k,
    precision_at_k,
    recall_at_k
)

# Constants TODO import these constants from constants
EVALUATION_USER_NUMBER = 200
EVALUATION_USER_NUMBER_DIVERSIFICATION = 100
NUMBER_OF_PRESELECTED_OFFERS = 50
RECOMMENDATION_NUMBER = 40
SHUFFLE_RECOMMENDATION = True

from commons.data_collect_queries import read_from_gcs
from commons.mlflow_tools import connect_remote_mlflow

def load_model(experiment_name, run_id):
    """Load a TensorFlow model from MLflow."""
    connect_remote_mlflow()
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
    
    with mlflow.start_run(experiment_id=experiment_id, run_id=run_id) as run:
        artifact_uri = mlflow.get_artifact_uri("model")
        loaded_model = tf.keras.models.load_model(
            artifact_uri,
            compile=False,
        )
    return loaded_model

def prepare_test_data(storage_path, test_dataset_name, raw_data, prediction_input_feature, n_users=None):
    """Prepare test data for evaluation.
    get full dataset and filter it to get only the users till n_users
    """
    test_columns = [prediction_input_feature, "item_id"]
    if "user_id" not in test_columns:
        test_columns.append("user_id")
    
    data_test = read_from_gcs(
        storage_path, test_dataset_name, parallel=False
    ).astype({"user_id": str, "item_id": str})[test_columns]

    data_test = data_test.merge(
        raw_data, on=["user_id", "item_id"], how="inner"
    ).drop_duplicates()

        
    if n_users:
        users_to_test = data_test["user_id"].unique()[
            :min(n_users, data_test["user_id"].nunique())
        ] # TODO : chose random or change this to chose the users with the most interactions
    else:
        users_to_test = data_test["user_id"].unique()
        
    return data_test, users_to_test


def generate_predictions(model, data_test, prediction_input_feature_names="user_id", users_to_test=None):
    """
    Generate recommendations for each user in the test set.
    in the format of a DataFrame with columns: user_id, item_id, score
    """
    
    # Get unique items to score
    data = data_test[["item_id", "offer_subcategory_id"]].drop_duplicates()
    offer_to_score = data.item_id.to_numpy() #.reshape(-1, 1)
    
    # Create empty DataFrame to store predictions
    df_predictions = pd.DataFrame(columns=["user_id", "item_id", "score"])

    if users_to_test is not None:
        data_test = data_test[data_test.user_id.isin(users_to_test)]
        
    for current_user in tqdm(data_test["user_id"].unique()):
        if prediction_input_feature_names != "user_id":
            input_feature = (
                data_test.loc[data_test["user_id"] == current_user, prediction_input_feature_names]
                .drop_duplicates()
                .tolist()[0]
            )
        else:
            input_feature = current_user
        
        # Prepare input for prediction
        prediction_input = [
            np.array([input_feature] * len(offer_to_score)),
            offer_to_score,
        ]
        
        # Generate predictions
        prediction = model.predict(prediction_input, verbose=0)
        
        # Append predictions to final predictions DataFrame
        user_predictions = pd.DataFrame(
        {
            "user_id" : [current_user]*len(offer_to_score),
            "item_id": offer_to_score.flatten().tolist(),
            "score": prediction.flatten().tolist(), 
            }
        )
        df_predictions = pd.concat([df_predictions, user_predictions])

    return df_predictions

def evaluate_recommendations(data_test, df_predictions,  k=10):
    """Evaluate recommendation quality using various metrics."""
    
    # Column definitions
    col_user = "user_id"
    col_item = "item_id"
    col_prediction = "score"
    
    # Calculate metrics
    metrics = {
        f"precision@{k}": precision_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,
        ),
        f"recall@{k}": recall_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,
        ),
        f"RPrecision@{k}": r_precision_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,
        ),
        f"map@{k}": map_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,
        )
    }
    return metrics

users=[]
def main(k=26):
    # # Configuration dev
    # STORAGE_PATH = "gs://mlflow-bucket-ehp/algo_training_stg/algo_training_two_towers_20250219T120000"
    # experiment_name = "algo_training_two_towers_cibrahim"
    # run_id = "eb3c90db3f424aa4abea00b76102144e"
    
    # Configuration prod
    STORAGE_PATH = "gs://mlflow-bucket-prod/algo_training_prod/algo_training_two_towers_20250227T132939"
    experiment_name = "algo_training_two_towers_v1.2_prod"
    run_id = "e6e14101ec2b463296fe7b1182375d6c"

    test_dataset_name = "recommendation_test_data"
    prediction_input_feature = "user_id"
    
    # Load model
    model = load_model(experiment_name, run_id)
    
    # Load raw data (bookings)
    raw_data = read_from_gcs(STORAGE_PATH, "bookings", parallel=False).astype(
        {"user_id": str, "item_id": str}
    )
    
    # Prepare test data
    data_test, users_to_test = prepare_test_data(
        STORAGE_PATH, test_dataset_name, raw_data, prediction_input_feature, 
    )
    users.append(users_to_test)
    
    # Generate predictions
    df_predictions = generate_predictions(
        model,
        data_test,
        prediction_input_feature,
    )
    
    metrics = evaluate_recommendations(data_test=data_test, df_predictions=df_predictions, k=k)
    
    print("Evaluation metrics:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")
    
    return metrics


In [2]:
import numpy as np
import pandas as pd
import pytest
from recommenders.evaluation.python_evaluation import (
    precision_at_k,
    recall_at_k,
    r_precision_at_k,
    map_at_k,
)

def test_metrics_computation():
    """
    Perform sanity checks on precision@k, recall@k, and other metrics.
    """
    test_cases = [
        {
            "name": "Perfect Recommendations",
            "data_test": pd.DataFrame({"user_id": ["A", "A"], "item_id": ["1", "2"]}),
            "df_predictions": pd.DataFrame({
                "user_id": ["A", "A"],
                "item_id": ["1", "2"],
                "score": [1.0, 0.9],
            }),
            "expected_precision": 1.0,
            "expected_recall": 1.0,
        },
        {
            "name": "Random Recommendations",
            "data_test": pd.DataFrame({"user_id": ["A"], "item_id": ["1"]}),
            "df_predictions": pd.DataFrame({
                "user_id": ["A", "A"],
                "item_id": ["3", "4"],  # Wrong items
                "score": [0.8, 0.7],
            }),
            "expected_precision": 0.0,
            "expected_recall": 0.0,
        },
        {
            "name": "Empty Recommendations",
            "data_test": pd.DataFrame({"user_id": ["A"], "item_id": ["1"]}),
            "df_predictions": pd.DataFrame(columns=["user_id", "item_id", "score"]),
            "expected_precision": 0.0,
            "expected_recall": 0.0,
        },
    ]
    
    for case in test_cases:
        precision = precision_at_k(
            case["data_test"], case["df_predictions"], col_user="user_id", col_item="item_id", col_prediction="score", k=2
        )
        recall = recall_at_k(
            case["data_test"], case["df_predictions"], col_user="user_id", col_item="item_id", col_prediction="score", k=2
        )
        
        assert np.isclose(precision, case["expected_precision"]), f"{case['name']} failed: Precision@2 mismatch. Expected {case['expected_precision']}, got {precision}"
        assert np.isclose(recall, case["expected_recall"]), f"{case['name']} failed: Recall@2 mismatch. Expected {case['expected_recall']}, got {recall}"

def test_validate_evaluation_metrics():
    """
    Run additional consistency checks on real data.
    """
    data_test = pd.DataFrame({"user_id": ["A", "B", "C"], "item_id": ["1", "2", "3"]})
    df_predictions = pd.DataFrame({
        "user_id": ["A", "B", "C"],
        "item_id": ["1", "2", "3"],
        "score": [0.9, 0.8, 0.7],
    })
    k = 2
    
    metrics = {
        f"precision@{k}": precision_at_k(data_test, df_predictions, col_user="user_id", col_item="item_id", col_prediction="score", k=k),
        f"recall@{k}": recall_at_k(data_test, df_predictions, col_user="user_id", col_item="item_id", col_prediction="score", k=k),
    }
    
    recall_k_plus_5 = recall_at_k(data_test, df_predictions, col_user="user_id", col_item="item_id", col_prediction="score", k=k+5)
    assert metrics[f"recall@{k}"] <= recall_k_plus_5, f"Recall@{k} is higher than Recall@{k+5}, which is incorrect."




In [3]:
## Baselines: random and most popular
def generate_random_baseline(data_test, users_to_test=None,  num_recommendations=10):
    """
    Generate random recommendations for each user.
    """
    unique_items = data_test["item_id"].unique()
    df_random = data_test[["user_id"]].drop_duplicates().copy()
    if users_to_test is not None:
        df_random = df_random[df_random.user_id.isin(users_to_test)]
    df_random = df_random.loc[df_random.index.repeat(num_recommendations)]
    df_random["item_id"] = np.random.choice(unique_items, size=len(df_random), replace=True)
    df_random["score"] = np.random.rand(len(df_random))
    return df_random

def generate_popularity_baseline(raw_data, data_test, users_to_test=None, num_recommendations=10, quantile_threshold=0.9):
    """
    Recommend the most popular items based on past bookings.

    Parameters:
        raw_data (pd.DataFrame): Historical bookings data.
        data_test (pd.DataFrame): Users to generate recommendations for.
        num_recommendations (int): Number of items to recommend per user.
        quantile_threshold (float): Percentage (0-1) of top popular items to consider.

    Returns:
        pd.DataFrame: DataFrame with columns ['user_id', 'item_id', 'score'].
    """
    # Compute item popularity
    item_popularity = raw_data["item_id"].value_counts().reset_index()
    item_popularity.columns = ["item_id", "popularity"]

    # Compute popularity threshold (top q%)
    popularity_cutoff = item_popularity["popularity"].quantile(quantile_threshold)

    # Filter items above the popularity threshold
    top_items = item_popularity[item_popularity["popularity"] >= popularity_cutoff]["item_id"].tolist()

    # If not enough items, take all available ones
    if len(top_items) < num_recommendations:
        top_items = item_popularity["item_id"].tolist()[:num_recommendations]

    # Generate recommendations for each user
    df_popular = data_test[["user_id"]].drop_duplicates().copy()
    if users_to_test is not None:
        df_popular = df_popular[df_popular.user_id.isin(users_to_test)]
    df_popular = df_popular.loc[df_popular.index.repeat(num_recommendations)].reset_index(drop=True)
    
    # Assign item recommendations
    df_popular["item_id"] = (top_items * (len(df_popular) // len(top_items) + 1))[:len(df_popular)]

    # Dynamically assign decreasing scores to match df_popular's length
    scores = np.linspace(1, 0.5, num=num_recommendations)  # Create a base score list
    df_popular["score"] = np.tile(scores, len(df_popular) // num_recommendations)[:len(df_popular)]

    return df_popular


In [4]:

# Configuration stg
STORAGE_PATH = "gs://mlflow-bucket-ehp/algo_training_stg/algo_training_two_towers_20250219T120000"
experiment_name = "algo_training_two_towers_cibrahim"
run_id = "eb3c90db3f424aa4abea00b76102144e"

# Configuration prod
STORAGE_PATH = "gs://mlflow-bucket-prod/algo_training_prod/algo_training_two_towers_20250227T132939"
experiment_name = "algo_training_two_towers_v1.2_prod"
run_id = "e6e14101ec2b463296fe7b1182375d6c"

test_dataset_name = "recommendation_test_data"
prediction_input_feature = "user_id"

# Load model
model = load_model(experiment_name, run_id)

# Load raw data (bookings)
raw_data = read_from_gcs(STORAGE_PATH, "bookings", parallel=False).astype(
    {"user_id": str, "item_id": str}
)

E0000 00:00:1741611840.375773  318630 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611840.393891  318628 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611840.412859  318633 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611840.433926  318629 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611840.454975  318628 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611840.474344  318629 ssl_transpor

RetryError: Deadline of 60.0s exceeded while calling target function, last exception: 503 failed to connect to all addresses; last error: UNKNOWN: ipv6:%5B2a00:1450:4007:807::200a%5D:443: Ssl handshake failed (TSI_PROTOCOL_FAILURE): SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED

E0000 00:00:1741611890.368579  318632 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611890.368614  318627 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611890.368662  318626 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611890.368717  318633 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611890.368875  318627 ssl_transport_security.cc:1654] Handshake failed with fatal error SSL_ERROR_SSL: error:1000007d:SSL routines:OPENSSL_internal:CERTIFICATE_VERIFY_FAILED
E0000 00:00:1741611890.368946  318633 ssl_transpor

In [94]:
# Prepare test data
data_test, users_to_test = prepare_test_data(
    STORAGE_PATH, test_dataset_name, raw_data, prediction_input_feature, n_users=500) #

# # If you want ot test users which have specific counts
# counts = data_test['user_id'].value_counts()
# list_users = [np.random.choice(data_test[data_test['user_id'].isin(counts.index[(counts > 1) & (counts <4)])]['user_id'].to_list() ) for i in range(10) ]
# data_test = data_test[data_test['user_id'].isin(list_users)]
# users_to_test = list_users

# Generate predictions
df_predictions = generate_predictions(
    model,
    data_test,
    prediction_input_feature,
    users_to_test
)
test_metrics_computation()
test_validate_evaluation_metrics()
print("All tests passed successfully! 🎯")

  0%|          | 0/500 [00:00<?, ?it/s]/var/folders/dw/nz0r7_t973z5_j607rfp9cvh0000gp/T/ipykernel_1326/1252045899.py:108: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_predictions = pd.concat([df_predictions, user_predictions])
100%|██████████| 500/500 [03:51<00:00,  2.16it/s]

All tests passed successfully! 🎯


In [95]:
print(f"Number of users in test {data_test.user_id.nunique()}")
print(f'Number of items to select from {data_test.item_id.nunique()}')

Number of users in test 122141
Number of items to select from 29317


In [96]:
# This cell shows the difference of metrics when varying k, on the same dataset, same nb of users 
for k in [10, 40, 100, 250, 1000]: 
    metrics = evaluate_recommendations(data_test=data_test, df_predictions=df_predictions, k=k)
    print(f"Evaluation metrics for {k}:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")
    print()

Evaluation metrics for 10:
precision@10: 0.0244
recall@10: 0.2397
RPrecision@10: 0.9706
map@10: 0.0916

Evaluation metrics for 40:
precision@40: 0.0116
recall@40: 0.4517
RPrecision@40: 0.9706
map@40: 0.1024

Evaluation metrics for 100:
precision@100: 0.0057
recall@100: 0.5533
RPrecision@100: 0.9706
map@100: 0.1041

Evaluation metrics for 250:
precision@250: 0.0027
recall@250: 0.6540
RPrecision@250: 0.9706
map@250: 0.1048

Evaluation metrics for 1000:
precision@1000: 0.0008
recall@1000: 0.7737
RPrecision@1000: 0.9706
map@1000: 0.1051



In [97]:
# Random Baseline
for k in [10, 40, 100, 250, 1000]: 
    df_random = generate_random_baseline(data_test, num_recommendations=k)
    metrics = evaluate_recommendations(data_test=data_test, df_predictions=df_random, k=k)
    print(f"Random baseline Evaluation metrics for {k}:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")
    print()

Random baseline Evaluation metrics for 10:
precision@10: 0.0000
recall@10: 0.0003
RPrecision@10: 1.0000
map@10: 0.0001

Random baseline Evaluation metrics for 40:
precision@40: 0.0000
recall@40: 0.0012
RPrecision@40: 1.0000
map@40: 0.0001

Random baseline Evaluation metrics for 100:
precision@100: 0.0000
recall@100: 0.0031
RPrecision@100: 1.0000
map@100: 0.0002

Random baseline Evaluation metrics for 250:
precision@250: 0.0000
recall@250: 0.0087
RPrecision@250: 1.0000
map@250: 0.0002



KeyboardInterrupt: 

In [98]:
# Popular Baseline
for k in [10, 40, 100, 250, 1000]: 
    df_popular = generate_popularity_baseline(raw_data, data_test, num_recommendations=1000, quantile_threshold=0.99)
    metrics = evaluate_recommendations(data_test=data_test, df_predictions=df_popular, k=k)
    print(f"Popular baseline evaluation metrics for {k}:")
    for metric_name, value in metrics.items():
        print(f"{metric_name}: {value:.4f}")
    print()

KeyboardInterrupt: 

In [117]:
%%time
# Column definitions
col_user = "user_id"
col_item = "item_id"
col_prediction = "score"
print(k, data_test.shape, df_predictions.shape)
p = precision_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,)

recall_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,
        ),
r_precision_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,
        )
map_at_k(
            data_test,
            df_predictions,
            col_user=col_user,
            col_item=col_item,
            col_prediction=col_prediction,
            relevancy_method="top_k",
            k=k,
        )

10 (128741, 15) (14660500, 3)
CPU times: user 59 s, sys: 6.94 s, total: 1min 5s
Wall time: 1min 7s


0.0915809523809524

In [116]:
df_predictions[df_predictions.user_id == "2542026"]

,user_id,item_id,score
0,2542026,product-1818360,0.656063
1,2542026,link-b7d1bb2e-1877-4492-aebb-d8eafa5b246a,0.195194
2,2542026,link-15452117-9043-4659-a481-eb9f9adc92ed,0.319365
3,2542026,product-6306154,-0.148980
4,2542026,product-6325361,0.550041
...,...,...,...
29316,2542026,product-4586686,-0.022271
29317,2542026,product-6207600,0.104279
29318,2542026,product-3096718,0.187314
29319,2542026,product-5911015,-0.003889


In [ ]:
# This cell shows how for the same k, calling the main function from scartch changes the values (i.e. how we can evaluate on different datasets)
k=10
for i in range(5):
    main(k)
    print()

KeyboardInterrupt: 

### RPrecision@k

* `df_hit` : dataframe with shape (nb of TP, 3) 
    * rank = rank of the hit in the list of recommendations of the user
* `df_hit_count` : dataframe with shape (nb of users, 3) 
    * hit = number of TP per user
    * actual = number of relevant items per user = TP+FN
* `df_merged` : 
    * dataframe originally with shape (nb of users, 4) with following columns:
        * `rank`: rank of reco in list of recos
        * `actual`: number of relevant items of user
    * Then filter the dataframe to keep only predictions for which $$rank ≤ \text{number of relevant items}$$

* RPrecision@k:
$$ 
RPrecision = \frac{\sum_{users} \frac{\text{\# recos whose rank is lesser than number of relevant items }}{\text{number of relevant items}} }{\text{number of users}}
$$

In [97]:
# this cell is to test r precision at k
from recommenders.evaluation.python_evaluation import merge_ranking_true_pred

#### Inside RPrecisio@k ####
col_user = "user_id"
df_hit, df_hit_count, n_users = merge_ranking_true_pred(
        rating_true=data_test,
        rating_pred=df_predictions,
        col_user="user_id",
        col_item="item_id",
        col_prediction="score",
        relevancy_method="top_k",
        k=k,
    )
df_merged = df_hit.merge(df_hit_count[["user_id", 'actual']])
df_merged = df_merged[df_merged['rank'] <= df_merged['actual']]
rprecsion= (df_merged.groupby(col_user).size() /  # number of recos whose rank is lesser than the number of relevant items
            df_hit_count.set_index(col_user)['actual'] # number of relevant items per user
            ).mean() # 


In [98]:
df_hit_count.set_index(col_user)['actual']

user_id
2713006    2
2995927    2
3171609    3
3587734    2
3840774    2
4108111    2
4879653    2
6944047    2
7374584    2
Name: actual, dtype: int64

### MAP@k
* hit = True Positive recommendation = TP
* relevant item = item that was consumed by user = TP+FN

$$ rr_{user} = \sum_{\text{hits in recos}} \frac{\text{rank of the hit in the list of hits}}{\text{rank of the hit in the list of all recommendations}}$$

$$ MAP@k = \frac{\sum_{\text{ users}} \frac{rr_{user}}{min(k, \text{number of relevant items of the user})}}{\text{number of users}} $$

In [ ]:
from recommenders.evaluation.python_evaluation import _get_reciprocal_rank

#### inside _get_reciprocal_rank: ####
col_user, col_item , col_prediction = "user_id", "item_id", "score"
relevancy_method = "top_k"

df_hit, df_hit_count, n_users = merge_ranking_true_pred(
        rating_true=data_test,
        rating_pred=df_predictions,
        col_user=col_user,
        col_item=col_item,
        col_prediction=col_prediction,
        relevancy_method=relevancy_method,
        k=k,
    )

# calculate reciprocal rank of items for each user and sum them up
df_hit_sorted = df_hit.copy()
df_hit_sorted["rr"] = (
    df_hit_sorted.groupby(col_user).cumcount() + 1
) / df_hit_sorted["rank"]
df_hit_sorted = df_hit_sorted.groupby(col_user).agg({"rr": "sum"}).reset_index()

df_merge, n_users = pd.merge(df_hit_sorted, df_hit_count, on=col_user), n_users

####  Inside map@k  ####
(
df_merge["rr"] / df_merge["actual"].apply(lambda x: min(x, k))
).sum() / n_users


0.51

In [76]:
df_merge

,user_id,rr,hit,actual
0,2713006,0.111111,1,2
1,2995927,1.000000,1,2
2,3171609,1.033333,3,3
3,3587734,2.000000,2,2
4,3840774,0.700000,2,2
5,4108111,3.000000,2,2
6,4879653,1.000000,2,2
7,6944047,0.450000,2,2
8,7374584,1.250000,2,2


### Novelty
$$ 
\text{item novelty }= -ln(\frac{\text{number of interactions with item}}{\text{number of interactions with all items}})
$$

$$
\text{novelty = the average novelty in a list of recommended items }
$$

Novelty is computed as the minus logarithm of (number of interactions with item / total number of interactions). The definition of the metric is based on the following reference using the choice model (eqs. 1 and 6):

:Citation:

    P. Castells, S. Vargas, and J. Wang, Novelty and diversity metrics for recommender systems:
    choice, discovery and relevance, ECIR 2011

The novelty of an item can be defined relative to a set of observed events on the set of all items.
These can be events of user choice (item "is picked" by a random user) or user discovery
(item "is known" to a random user). The above definition of novelty reflects a factor of item popularity.
High novelty values correspond to long-tail items in the density function, that few users have interacted with and low novelty values correspond to popular head items.

In [40]:
from recommenders.evaluation.python_evaluation import novelty
# Perform an anti-join: Keep only (user_id, item_id) pairs that are NOT in raw_data
reco_df = df_predictions.merge(raw_data, on=['user_id', 'item_id'], how='left', indicator=True)
reco_df = reco_df[reco_df['_merge'] == 'left_only'].drop(columns=['_merge'])

novelty(raw_data[['user_id', 'item_id']].drop_duplicates(),
        reco_df[['user_id', 'item_id']], 
        col_item="item_id",
        col_user="user_id"
        ) 

/Users/Carole/projects/data-gcp/jobs/ml_jobs/algo_training/.venv/lib/python3.10/site-packages/recommenders/evaluation/python_evaluation.py:1435: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  avg_novelty = reco_item_novelty.agg({"product": "sum"})[0] / n_recommendations


17.98529665206926

### Catalog coverage 
$$
coverage = \frac{\text{number of unique items recommended for all users}}{\text{number of items in catalogue}}
$$

In [41]:
from recommenders.evaluation.python_evaluation import catalog_coverage
catalog_coverage(raw_data[['user_id', 'item_id']].drop_duplicates(), reco_df, col_user="user_id", col_item="item_id")

0.07090632490126034

### Distributional coverage

In [42]:
from recommenders.evaluation.python_evaluation import distributional_coverage


#### Inside distributional coverage ####
col_item="item_id"
col_user="user_id"
# In reco_df, how  many times each col_item is being recommended
df_itemcnt_reco = pd.DataFrame(
    {"count": reco_df.groupby([col_item]).size()}
).reset_index()

# the number of total recommendations
count_row_reco = reco_df.shape[0]

df_entropy = df_itemcnt_reco
df_entropy["p(i)"] = df_entropy["count"] / count_row_reco
df_entropy["entropy(i)"] = df_entropy["p(i)"] * np.log2(df_entropy["p(i)"])

d_coverage = -df_entropy.agg({"entropy(i)": "sum"})[0]

d_coverage

/var/folders/dw/nz0r7_t973z5_j607rfp9cvh0000gp/T/ipykernel_1326/880997970.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  d_coverage = -df_entropy.agg({"entropy(i)": "sum"})[0]


14.839372485982825